In [19]:
#Install required frameworks
!pip install -q transformers accelerate bitsandbytes sentence-transformers scikit-learn feedparser streamlit
!npm install -g localtunnel

#Create the offload directory
import os
if not os.path.exists("offload"):
    os.makedirs("offload")
#Clear background processes using GPU
import torch
torch.cuda.empty_cache()
!pkill -9 streamlit
!pkill -9 lt

⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 1s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [20]:
!pip install -q rouge-score

In [25]:
%%writefile app.py
import streamlit as st
import feedparser
import torch
import time
import re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.cluster import AgglomerativeClustering
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from google.colab import userdata
#Page Configuration
st.set_page_config(page_title="News Summarizer Dashboard", layout="wide")

st.html("""
<style>
    /* Hides the 'arrow' and 'keyboard_double_arrow' text leakage */
    span[data-testid="stHeaderActionElements"],
    .st-emotion-cache-p997v8,
    .st-emotion-cache-6qob1r {
        display: none !important;
    }
    /* Specifically target the text inside expanders and sidebars */
    div[data-testid="stExpander"] svg + span,
    div[data-sidebar-user-content] span {
        font-size: 0 !important;
    }
</style>
""")

st.title("📰 Real-Time News Summarizer & Evaluation")

def clean_html(text):
    if not text:
        return ""
    return re.sub(re.compile('<.*?>'), '', text)

@st.cache_resource
def load_llm():
    model_id = "mistralai/Mistral-7B-Instruct-v0.3"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16,
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, quantization_config=bnb_config, device_map="auto",token=True
    )
    return tokenizer, model

@st.cache_resource
def load_embedder():
    return SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

tokenizer, model = load_llm()
embed_model = load_embedder()
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

#Sidebar
st.sidebar.markdown("### ⚙️ Settings")
RSS_FEEDS = {
    "Google News": "https://news.google.com/rss?hl=en-US&gl=US&ceid=US:en",
    "BBC World": "http://feeds.bbci.co.uk/news/world/rss.xml"
}
selected_feed = st.sidebar.selectbox("News Source", list(RSS_FEEDS.keys()))
cluster_threshold = st.sidebar.slider("Clustering Sensitivity", 0.1, 1.5, 0.6)

if st.button("🚀 Fetch, Summarize & Evaluate"):
    with st.spinner("Processing news clusters..."):
        feed = feedparser.parse(RSS_FEEDS[selected_feed])
        articles = feed.entries[:12]

        if articles:
            #Clustering
            embeddings = embed_model.encode([clean_html(f"{a.title} {a.summary}") for a in articles])
            labels = AgglomerativeClustering(n_clusters=None, distance_threshold=cluster_threshold).fit_predict(embeddings)

            clusters = {}
            for idx, label in enumerate(labels):
                if label not in clusters: clusters[label] = []
                clusters[label].append(articles[idx])

            #Generation & Evaluation
            for c_id, cluster in clusters.items():
                start_time = time.time()

                context_data = "\n".join([f"TITLE: {a.title}\nTEXT: {clean_html(a.summary)}" for a in cluster])
                prompt = f"<s>[INST] Generate a 5-word headline and a 3-sentence summary for these reports. Start the headline with 'TOPIC:' and the summary with 'REPORT:'.\n\n{context_data} [/INST]"

                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

                with torch.no_grad():
                    outputs = model.generate(**inputs, max_new_tokens=250, temperature=0.1)

                #Logic to only show model response
                input_length = inputs.input_ids.shape[1]
                generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()

                # Extraction
                headline = "News Update"
                summary = generated_text

                topic_match = re.search(r"TOPIC:(.*?)(?=REPORT:|$)", generated_text, re.DOTALL | re.IGNORECASE)
                report_match = re.search(r"REPORT:(.*)", generated_text, re.DOTALL | re.IGNORECASE)

                if topic_match:
                    headline = topic_match.group(1).strip()
                if report_match:
                    summary = report_match.group(1).strip()

                #EVALUATION CALCULATIONS
                latency = round(time.time() - start_time, 2)

                # Compression Ratio (Summary Words / Original Words)
                original_word_count = sum(len(clean_html(a.summary).split()) for a in cluster)
                summary_word_count = len(summary.split())
                comp_ratio = round(summary_word_count / max(original_word_count, 1), 2)

                # Self-ROUGE (Similarity to the first article as a proxy reference)
                ref_text = clean_html(cluster[0].summary)
                rouge_scores = scorer.score(ref_text, summary)
                rouge_l = round(rouge_scores['rougeL'].fmeasure, 2)

                #DASHBOARD UI
                with st.container(border=True):
                    st.subheader(f"📌 {headline}")

                    # Evaluation Metrics Row
                    m1, m2, m3 = st.columns(3)
                    m1.metric("Latency", f"{latency}s")
                    m2.metric("Comp. Ratio", f"{comp_ratio}x", help="Lower is more compressed")
                    m3.metric("ROUGE-L", f"{rouge_l}", help="Similarity to lead source")

                    st.markdown(f"**Summary:** {summary}")

                    with st.expander("🔗 Sources"):
                        for a in cluster:
                            st.markdown(f"• [{a.title}]({a.link})")
        else:
            st.error("Could not fetch news. Check your connection or the RSS feed link.")

Overwriting app.py


In [29]:
import os
import subprocess
import time
from pyngrok import ngrok
from google.colab import userdata


try:
    # Pulling tokens from Colab Secrets
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    print("✅ Secrets loaded successfully.")
except Exception as e:
    print("❌ Error: Make sure you added 'HF_TOKEN' and 'NGROK_AUTH_TOKEN' to the Secrets (key icon) sidebar.")
    raise e

#Cleanup
!pkill -9 streamlit
!pkill -9 ngrok
ngrok.kill()

#Authenticate Ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
!ngrok config add-authtoken {NGROK_AUTH_TOKEN}

#Start Streamlit
with open("streamlit_logs.txt", "w") as f:
    subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501"],
        stdout=f,
        stderr=f,
        env=os.environ
    )

#Connect Tunnel
time.sleep(5)
try:
    public_url = ngrok.connect(8501)
    print(f"\n🚀 Streamlit is live at: {public_url}")
except Exception as e:
    print(f"Tunnel error: {e}")

#loading progress
print("Monitoring logs (Model download/loading)...")
!tail -f streamlit_logs.txt

✅ Secrets loaded successfully.
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

🚀 Streamlit is live at: NgrokTunnel: "https://arson-wrecker-simple.ngrok-free.dev" -> "http://localhost:8501"
Monitoring logs (Model download/loading)...



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.240.139.129:8501

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 584.49it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting 

^C
